# H-003 Baseline Experiment

WP-3b baseline notebook. This notebook reproduces the baseline metrics from repo-truth data and helpers. It does not append to `research/trial_log.jsonl` when rerun. The committed trial ID is `b56ccac3-306d-45b0-9626-6b37d4065002`.


Evaluation note: no separate IS/OOS split convention exists in repo truth. The baseline uses the full registered evaluation range for both `in_sample_sharpe` and `oos_sharpe`, matching the current runbook baseline pattern.

Gate 2 semantics note: this notebook reports Gate 2 according to current `lib/evaluation.py` repo truth. The displayed legacy `GateReport` wording may read `DSR > 1.0 at 95% CI`, but the evaluator currently checks whether `dsr_val` exceeds 0.95 against its expected maximum-SR benchmark from the supplied trial count. Interpret the Gate 2 result according to the evaluator implementation, not as proof that the strategy met a true Sharpe > 1.0 standard.

Data note: WP-3b uses the catalogued 1d OHLCV parquet directly; no derived 1d data is persisted. Funding is passed because `run_backtest()` requires a funding input and models funding cashflows for open perpetual positions.


In [1]:
from pathlib import Path
import json
import math
import pandas as pd
from lib.backtest import run_backtest
from lib.evaluation import check_promotion_gates, cost_drag, hit_rate, max_drawdown, profit_factor, sharpe
from lib.evaluation import _sha256_of_dict

OHLCV_PATH = Path("data/processed/binance_um_perp_BTCUSDT_1d_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet")
FUNDING_PATH = Path("data/processed/binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet")
DATASET_IDS = [
    "binance_um_perp_BTCUSDT_1d_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z",
]

def load_ohlcv():
    df = pd.read_parquet(OHLCV_PATH).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    return df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df["fundingTime"].astype("int64"), unit="ms", utc=True)
    return df.rename(columns={"fundingRate": "funding_rate"})[["funding_rate"]].astype(float).sort_index()

def h003_signal(ohlcv, bb_period=20, bb_std_dev=2.0):
    lb = int(round(bb_period))
    middle = ohlcv["close"].rolling(lb, min_periods=lb).mean()
    lower = middle - float(bb_std_dev) * ohlcv["close"].rolling(lb, min_periods=lb).std()
    signal = pd.Series(0.0, index=ohlcv.index, name="signal")
    current = 0.0
    for ts in ohlcv.index:
        if pd.isna(middle.loc[ts]) or pd.isna(lower.loc[ts]):
            current = 0.0
        elif ohlcv.loc[ts, "close"] < lower.loc[ts]:
            current = 1.0
        elif current == 1.0 and ohlcv.loc[ts, "close"] >= middle.loc[ts]:
            current = 0.0
        signal.loc[ts] = current
    return signal

def h004_signal(ohlcv, fast_ma=50, slow_ma=200, rv_lookback=30, rv_percentile=50):
    fast = ohlcv["close"].rolling(int(round(fast_ma)), min_periods=int(round(fast_ma))).mean()
    slow = ohlcv["close"].rolling(int(round(slow_ma)), min_periods=int(round(slow_ma))).mean()
    rv = ohlcv["close"].pct_change().rolling(int(round(rv_lookback)), min_periods=int(round(rv_lookback))).std()
    threshold = rv.shift(1).rolling(252, min_periods=252).quantile(float(rv_percentile) / 100.0)
    high_vol = rv > threshold
    signal = pd.Series(0.0, index=ohlcv.index, name="signal")
    signal[(fast > slow) & high_vol] = 1.0
    signal[(fast < slow) & high_vol] = -1.0
    signal[fast.isna() | slow.isna() | rv.isna() | threshold.isna()] = 0.0
    return signal

def run_candidate(signal, ohlcv, funding):
    return run_backtest(signal, ohlcv, funding, fee_bps_taker=4.0, fee_bps_maker=2.0, slippage_bps=2.0, initial_equity=10000.0, execution="next_bar_open", stream="klines", interval="1d")

def summarize(result):
    returns = result.equity_curve.pct_change().dropna()
    mdd, peak, trough = max_drawdown(result.equity_curve)
    cd = cost_drag(result)
    return {
        "final_equity": float(result.equity_curve.iloc[-1]),
        "total_pnl": float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]),
        "sharpe": float(sharpe(returns, "1d")),
        "max_drawdown": float(mdd),
        "max_drawdown_peak": str(peak) if peak is not None else None,
        "max_drawdown_trough": str(trough) if trough is not None else None,
        "n_obs": int(len(returns)),
        "n_trades": int(len(result.trades_df)),
        "hit_rate": float(hit_rate(result.trades_df)),
        "profit_factor": float(profit_factor(result.trades_df)),
        "fees_paid": float(result.fees_paid),
        "slippage_paid": float(result.slippage_paid),
        "funding_paid": float(result.funding_paid),
        "gross_pnl": float(cd["gross_pnl"]),
        "cost_drag_fraction": float(cd["cost_drag_fraction"]) if math.isfinite(cd["cost_drag_fraction"]) else None,
        "skipped_bars": int(result.skipped_bars),
    }

def side_summary(result):
    if result.trades_df.empty:
        return []
    return [
        {"direction": int(d), "count": int(len(g)), "net_pnl_sum": float(g["net_pnl"].sum()), "net_pnl_mean": float(g["net_pnl"].mean())}
        for d, g in result.trades_df.groupby("direction")
    ]

def sensitivity_h003(ohlcv, funding, base):
    rows = []
    for name, params in [
        ("bb_period_up_25pct", {"bb_period": 25, "bb_std_dev": 2.0}),
        ("bb_period_down_25pct", {"bb_period": 15, "bb_std_dev": 2.0}),
        ("bb_std_dev_up_25pct", {"bb_period": 20, "bb_std_dev": 2.5}),
        ("bb_std_dev_down_25pct", {"bb_period": 20, "bb_std_dev": 1.5}),
    ]:
        sr = summarize(run_candidate(h003_signal(ohlcv, **params), ohlcv, funding))["sharpe"]
        rows.append({"variant": name, "sharpe": sr, "ratio_to_base": sr / base if base else 0.0})
    return rows

def sensitivity_h004(ohlcv, funding, base):
    variants = [
        ("fast_ma_up_25pct", {"fast_ma": 62.5, "slow_ma": 200, "rv_lookback": 30, "rv_percentile": 50}),
        ("fast_ma_down_25pct", {"fast_ma": 37.5, "slow_ma": 200, "rv_lookback": 30, "rv_percentile": 50}),
        ("slow_ma_up_25pct", {"fast_ma": 50, "slow_ma": 250, "rv_lookback": 30, "rv_percentile": 50}),
        ("slow_ma_down_25pct", {"fast_ma": 50, "slow_ma": 150, "rv_lookback": 30, "rv_percentile": 50}),
        ("rv_lookback_up_25pct", {"fast_ma": 50, "slow_ma": 200, "rv_lookback": 37.5, "rv_percentile": 50}),
        ("rv_lookback_down_25pct", {"fast_ma": 50, "slow_ma": 200, "rv_lookback": 22.5, "rv_percentile": 50}),
        ("rv_percentile_up_25pct", {"fast_ma": 50, "slow_ma": 200, "rv_lookback": 30, "rv_percentile": 62.5}),
        ("rv_percentile_down_25pct", {"fast_ma": 50, "slow_ma": 200, "rv_lookback": 30, "rv_percentile": 37.5}),
    ]
    rows = []
    for name, params in variants:
        sr = summarize(run_candidate(h004_signal(ohlcv, **params), ohlcv, funding))["sharpe"]
        rows.append({"variant": name, "sharpe": sr, "ratio_to_base": sr / base if base else 0.0})
    return rows

HYPOTHESIS_ID = "H-003"
TRIAL_ID = "b56ccac3-306d-45b0-9626-6b37d4065002"
TRIAL_COUNT_BEFORE_APPEND = 2
PARAMS = {'bb_period': 20, 'bb_std_dev': 2.0}
N_FREE = 2
SIGNAL_FN = h003_signal
SENS_FN = sensitivity_h003
ohlcv = load_ohlcv()
funding = load_funding()
params = PARAMS
signal = SIGNAL_FN(ohlcv, **params)
result = run_candidate(signal, ohlcv, funding)
metrics = summarize(result)
sensitivity = SENS_FN(ohlcv, funding, metrics["sharpe"])
params_meta = {"n_free_params": N_FREE, "sensitivity_min_ratio": min(r["ratio_to_base"] for r in sensitivity), "is_sharpe": metrics["sharpe"]}
gate_report = check_promotion_gates(result=result, trial_count=TRIAL_COUNT_BEFORE_APPEND, params_meta=params_meta, freq="1d")
config = {"hypothesis_id": HYPOTHESIS_ID, "primary_timeframe": "1d", "execution": "next_bar_open", "costs": {"fee_bps_taker": 4.0, "fee_bps_maker": 2.0, "slippage_bps": 2.0, "initial_equity": 10000.0}, "params": params}
print(json.dumps({
    "hypothesis_id": HYPOTHESIS_ID,
    "trial_id": TRIAL_ID,
    "trial_count_before_append": TRIAL_COUNT_BEFORE_APPEND,
    "dataset_ids": DATASET_IDS,
    "dataset_sha256": _sha256_of_dict({"ids": sorted(DATASET_IDS)}),
    "config_sha256": _sha256_of_dict(config),
    "params": params,
    "metrics": metrics,
    "sensitivity": sensitivity,
    "sensitivity_min_ratio": params_meta["sensitivity_min_ratio"],
    "side_summary": side_summary(result),
    "gate_report": gate_report.to_dict(),
}, indent=2, sort_keys=True))


{
  "config_sha256": "4b98a2d3bf3190a76c07a717e881ccc5655ec2d6d669ebdef6aae0a169eae6f7",
  "dataset_ids": [
    "binance_um_perp_BTCUSDT_1d_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z"
  ],
  "dataset_sha256": "47b91ceac5ae9e472a08c23390ec8a5f6fee9a627a44df11e3557fbace02813d",
  "gate_report": {
    "gates": [
      {
        "gate_id": 1,
        "gate_name": "OOS Sharpe >= 1.0",
        "note": "",
        "passed": false,
        "threshold": 1.0,
        "value": -0.4064
      },
      {
        "gate_id": 2,
        "gate_name": "DSR > 1.0 at 95% CI",
        "note": "P(SR>1.0)=0.0000; SR benchmark=0.1644",
        "passed": false,
        "threshold": 1.0,
        "value": 0.0
      },
      {
        "gate_id": 3,
        "gate_name": "Per-fold max DD <= 20%",
        "note": "",
        "passed": false,
        "threshold": 0.2,
        "value": 1.0959
      },
      {
        "gate_id": 4,
        "gat